In [6]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

from torchvision.models.detection import (
    maskrcnn_resnet50_fpn,
    MaskRCNN_ResNet50_FPN_Weights
)

from torchvision.models.segmentation import (
    deeplabv3_resnet50,
    DeepLabV3_ResNet50_Weights,
    fcn_resnet50,
    FCN_ResNet50_Weights
)

In [7]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available!")

device = torch.device("cuda:0")

IMAGE_SIZE = (256, 256)

DATA_DIR = "./data"

UNET_PATH = "unet_oxford_pet.pth"

SCORE_THRESHOLD = 0.5

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)

PyTorch: 2.7.1+cu128
Torchvision: 0.22.1+cu128
GPU: NVIDIA GeForce GTX 1650
Device: cuda:0


In [8]:
image_transform = transforms.Compose([

    transforms.Resize(
        IMAGE_SIZE,
        interpolation=InterpolationMode.BILINEAR
    ),

    transforms.ToTensor()

])


def mask_transform(mask):

    mask = TF.resize(
        mask,
        IMAGE_SIZE,
        interpolation=InterpolationMode.NEAREST
    )

    mask = TF.pil_to_tensor(mask)

    mask = (
        (mask == 1) | (mask == 3)
    ).float()

    return mask


test_dataset = datasets.OxfordIIITPet(

    root=DATA_DIR,
    split="test",
    target_types="segmentation",
    transform=image_transform,
    target_transform=mask_transform,
    download=False
)


test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


print("Test Samples:", len(test_dataset))

Test Samples: 3669


In [10]:
# Recreate U-Net architecture
class ConvBlock(nn.Module):

    def __init__(self,in_channels,out_channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace= True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace= True)
        )

    def forward(self,x):
        return self.block(x) 

class UNet(nn.Module):

    def __init__(self,in_channels=3,out_channels=1):

        super().__init__()

        self.pool = nn.MaxPool2d(kernel_size=2)

        # Encoder 
        self.enc1 = ConvBlock(in_channels,32)
        self.enc2 = ConvBlock(32,64)
        self.enc3 = ConvBlock(64,128)
        self.enc4 = ConvBlock(128,256)

        # BottleNeck 
        self.bottleneck = ConvBlock(256,512)

        # Decoder 
        self.up4 = nn.ConvTranspose2d(512,256,2,stride=2)
        self.dec4 = ConvBlock(512,256)

        self.up3 = nn.ConvTranspose2d(256,128,2,stride=2)
        self.dec3 = ConvBlock(256,128)

        self.up2 = nn.ConvTranspose2d(128,64,2,stride=2)
        self.dec2 = ConvBlock(128,64)

        self.up1 = nn.ConvTranspose2d(64,32,2,stride=2)
        self.dec1 = ConvBlock(64,32)

        # Final Segmentation Layer 
        self.out = nn.Conv2d(
            32,
            out_channels,
            kernel_size=1
        )

    def forward(self,x):

        # Encoder 
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        # Bottleneck
        b = self.bottleneck(self.pool(e4))

        # Decoder + Skip connections
        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)


# Initialize model from scratch
model = UNet().to(device)

print(model)

unet_model = UNet().to(device)

UNET_PATH = 'unet_oxford_pet.pth'

# Load existing checkpoint
checkpoint = torch.load(
    UNET_PATH,
    map_location="cpu",
    weights_only=True
)


unet_model.load_state_dict(checkpoint)

unet_model.eval()


print("U-Net Loaded Successfully!")

print(
    "U-Net Device:",
    next(unet_model.parameters()).device
)

UNet(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (enc1): ConvBlock(
    (block): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): ConvBlock(
    (block): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc

In [11]:
# --------------------------------
# MASK R-CNN
# --------------------------------

maskrcnn_weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT

maskrcnn_model = maskrcnn_resnet50_fpn(

    weights=maskrcnn_weights,

    min_size=256,

    max_size=256

).to(device)

maskrcnn_model.eval()


# --------------------------------
# DEEPLABV3
# --------------------------------

deeplab_weights = DeepLabV3_ResNet50_Weights.DEFAULT

deeplab_model = deeplabv3_resnet50(

    weights=deeplab_weights

).to(device)

deeplab_model.eval()


# --------------------------------
# FCN
# --------------------------------

fcn_weights = FCN_ResNet50_Weights.DEFAULT

fcn_model = fcn_resnet50(

    weights=fcn_weights

).to(device)

fcn_model.eval()


print("All pretrained models loaded successfully!")

print("Mask R-CNN:", next(maskrcnn_model.parameters()).device)

print("DeepLabV3:", next(deeplab_model.parameters()).device)

print("FCN:", next(fcn_model.parameters()).device)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to C:\Users\yuvan/.cache\torch\hub\checkpoints\maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:22<00:00, 7.87MB/s] 


Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to C:\Users\yuvan/.cache\torch\hub\checkpoints\deeplabv3_resnet50_coco-cd0a2569.pth


100%|██████████| 161M/161M [00:20<00:00, 8.22MB/s] 


Downloading: "https://download.pytorch.org/models/fcn_resnet50_coco-1167a1af.pth" to C:\Users\yuvan/.cache\torch\hub\checkpoints\fcn_resnet50_coco-1167a1af.pth


100%|██████████| 135M/135M [00:17<00:00, 8.15MB/s] 


All pretrained models loaded successfully!
Mask R-CNN: cuda:0
DeepLabV3: cuda:0
FCN: cuda:0


In [12]:
# --------------------------------
# CLASS INDICES
# --------------------------------

maskrcnn_categories = maskrcnn_weights.meta["categories"]

deeplab_categories = deeplab_weights.meta["categories"]

fcn_categories = fcn_weights.meta["categories"]


# Mask R-CNN class IDs
maskrcnn_cat_id = maskrcnn_categories.index("cat")

maskrcnn_dog_id = maskrcnn_categories.index("dog")


# DeepLabV3 class IDs
deeplab_cat_id = deeplab_categories.index("cat")

deeplab_dog_id = deeplab_categories.index("dog")


# FCN class IDs
fcn_cat_id = fcn_categories.index("cat")

fcn_dog_id = fcn_categories.index("dog")


print("Mask R-CNN:", maskrcnn_cat_id, maskrcnn_dog_id)

print("DeepLabV3:", deeplab_cat_id, deeplab_dog_id)

print("FCN:", fcn_cat_id, fcn_dog_id)


# --------------------------------
# PREPROCESSING
# --------------------------------

deeplab_preprocess = deeplab_weights.transforms()

fcn_preprocess = fcn_weights.transforms()

Mask R-CNN: 17 18
DeepLabV3: 8 12
FCN: 8 12


In [13]:
def predict_unet(image):

    image = image.unsqueeze(0).to(device)

    with torch.inference_mode():

        output = unet_model(image)

        prediction = (
            torch.sigmoid(output[0, 0]) >= 0.5
        )

    return prediction

In [14]:
def predict_maskrcnn(image):

    image = image.to(device)

    with torch.inference_mode():

        output = maskrcnn_model([image])[0]


    labels = output["labels"]

    scores = output["scores"]

    masks = output["masks"]


    # Select cat and dog detections
    selected = (

        (
            (labels == maskrcnn_cat_id) |
            (labels == maskrcnn_dog_id)
        )

        &

        (scores >= SCORE_THRESHOLD)

    )


    selected_masks = masks[selected]


    # Combine all detected pet masks
    prediction = torch.zeros(

        IMAGE_SIZE,

        dtype=torch.bool,

        device=device

    )


    for instance_mask in selected_masks:

        prediction |= (
            instance_mask[0] >= 0.5
        )


    return prediction

In [15]:
def predict_deeplab(image):

    # Apply pretrained preprocessing
    image = deeplab_preprocess(image)

    image = image.unsqueeze(0).to(device)


    with torch.inference_mode():

        output = deeplab_model(image)["out"]


        # Resize logits to original image size
        output = F.interpolate(

            output,

            size=IMAGE_SIZE,

            mode="bilinear",

            align_corners=False

        )


        # Predicted semantic class
        predicted_classes = output.argmax(dim=1)[0]


        # Combine cat and dog classes
        prediction = (

            (predicted_classes == deeplab_cat_id) |

            (predicted_classes == deeplab_dog_id)

        )


    return prediction

In [16]:
def predict_fcn(image):

    # Apply pretrained preprocessing
    image = fcn_preprocess(image)

    image = image.unsqueeze(0).to(device)


    with torch.inference_mode():

        output = fcn_model(image)["out"]


        # Resize logits to original image size
        output = F.interpolate(

            output,

            size=IMAGE_SIZE,

            mode="bilinear",

            align_corners=False

        )


        predicted_classes = output.argmax(dim=1)[0]


        # Combine cat and dog classes
        prediction = (

            (predicted_classes == fcn_cat_id) |

            (predicted_classes == fcn_dog_id)

        )


    return prediction

In [ ]:
predictors = {

    "U-Net": predict_unet,

    "Mask R-CNN": predict_maskrcnn,

    "DeepLabV3": predict_deeplab,

    "FCN": predict_fcn

}


MAX_TEST_IMAGES = 100

# Change to None for the complete test dataset.


stats = {

    name: {

        "intersection": 0,

        "prediction": 0,

        "target": 0,

        "time": 0.0,

        "count": 0

    }

    for name in predictors

}


sample_results = []


# --------------------------------
# WARM-UP
# --------------------------------

first_image, _ = test_dataset[0]

for predictor in predictors.values():

    _ = predictor(first_image)


torch.cuda.synchronize()


# --------------------------------
# EVALUATION
# --------------------------------

for idx, (images, masks) in enumerate(test_loader):

    if MAX_TEST_IMAGES is not None and idx >= MAX_TEST_IMAGES:

        break


    image = images[0]

    ground_truth = (
        masks[0, 0] > 0
    )


    sample = {

        "image": image.clone(),

        "ground_truth": ground_truth.clone(),

        "predictions": {}

    }


    for name, predictor in predictors.items():


        torch.cuda.synchronize()

        start_time = time.perf_counter()


        prediction = predictor(image)


        torch.cuda.synchronize()

        elapsed = time.perf_counter() - start_time


        prediction_cpu = prediction.cpu()

        intersection = (

            prediction_cpu & ground_truth

        ).sum().item()


        prediction_area = prediction_cpu.sum().item()

        target_area = ground_truth.sum().item()


        stats[name]["intersection"] += intersection

        stats[name]["prediction"] += prediction_area

        stats[name]["target"] += target_area

        stats[name]["time"] += elapsed

        stats[name]["count"] += 1


        # Store first four images for visualization
        if idx < 4:

            sample["predictions"][name] = prediction_cpu


    if idx < 4:

        sample_results.append(sample)


    if (idx + 1) % 10 == 0:

        print(
            f"Processed {idx + 1} images"
        )


print("\nEvaluation completed!")

Processed 10 images
